# Serenity AI — Evaluation Notebook

End-to-end evaluation of Serenity AI agents using [DeepEval](https://docs.confident-ai.com/).

**Workflow:**
1. Fill in **Configuration** (cell below)
2. Run **Setup** to load helpers
3. Run **Load Datasets** to see available test suites
4. Pick a dataset and run **Evaluate**
5. Inspect the **Results** table

---

## 0 · Install dependencies

Run once, then comment out.

In [ ]:
# pip install -r requirements.txt

## 1 · Configuration

Edit the variables below before running anything else.

In [ ]:
AI_SERVICE_URL     = "http://localhost:8001"
INTERNAL_API_TOKEN = "dev-internal-token"

OPENAI_API_KEY = "ENV_KEY"
ORG_ID = "cmqjbacig0001t7dqmyi90for"
AUTH_TOKEN   = "AuthToken"
ORG_SLUG     = "vku"
ORG_NAME     = "VKU"
USER_ID      = "cmqjbacid0000t7dqo7is0kg3"
DISPLAY_NAME = "Huy Phan"
EMAIL        = "huy.phn16@gmail.com"

## 2 · Setup

Imports, AI adapter, metric builder, evaluation runner, and results renderer.
Run this cell once — re-run if you change **Configuration**.

In [3]:
import io
import json
import logging
import os
import contextlib
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
from threading import Lock
from html import escape
from pathlib import Path
from uuid import uuid4

import httpx
from IPython.display import HTML, display
from tqdm.notebook import tqdm

# Silence DeepEval's banner/warning noise before importing it
logging.getLogger("deepeval").setLevel(logging.ERROR)
logging.getLogger("deepeval.metrics").setLevel(logging.ERROR)
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="deepeval")

# -----------------------------------------------------------------------------
# Runtime config
# -----------------------------------------------------------------------------
def _cfg(name: str, default: str = "") -> str:
    return str(globals().get(name) or os.getenv(name) or default)

OPENAI_API_KEY = _cfg("OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

AI_SERVICE_URL     = _cfg("AI_SERVICE_URL", "http://localhost:8000").rstrip("/")
INTERNAL_API_TOKEN = _cfg("INTERNAL_API_TOKEN")
AUTH_TOKEN         = _cfg("AUTH_TOKEN")
ORG_ID             = _cfg("ORG_ID")
USER_ID            = _cfg("USER_ID")
ORG_SLUG           = _cfg("ORG_SLUG")
ORG_NAME           = _cfg("ORG_NAME")
DISPLAY_NAME       = _cfg("DISPLAY_NAME")
EMAIL              = _cfg("EMAIL")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from deepeval import evaluate
    try:
        from deepeval.evaluate.configs import CacheConfig, DisplayConfig
    except Exception:
        CacheConfig = None
        DisplayConfig = None
    from deepeval.metrics import (
        AnswerRelevancyMetric,
        ContextualRecallMetric,
        FaithfulnessMetric,
        GEval,
    )
    from deepeval.test_case import LLMTestCase, LLMTestCaseParams


AGENT_DISPLAY_NAMES: dict[str, str] = {
    "wiki_agent":     "Wiki Agent",
    "chat_agent":     "Chat Agent",
    "calendar_agent": "Calendar Agent",
    "contacts_agent": "Contacts Agent",
    "mail_agent":     "Mail Agent",
    "schedule_agent": "Schedule Agent",
    "chat_assist":    "Chat Assist",
}

# Matches both the agent key AND the short feature name from dataset JSONs
FEATURE_DEFAULT_METRICS: dict[str, list[str]] = {
    "wiki_agent":     ["answer_relevancy", "extraction_accuracy"],
    "wiki":           ["answer_relevancy", "extraction_accuracy"],
    "chat_agent":     ["answer_relevancy", "faithfulness", "contextual_recall"],
    "chat":           ["answer_relevancy", "faithfulness", "contextual_recall"],
    "calendar_agent": ["answer_relevancy", "faithfulness", "contextual_recall", "extraction_accuracy"],
    "calendar":       ["answer_relevancy", "faithfulness", "contextual_recall", "extraction_accuracy"],
    "contacts_agent": ["answer_relevancy", "faithfulness", "contextual_recall"],
    "contacts":       ["answer_relevancy", "faithfulness", "contextual_recall"],
    "mail_agent":     ["answer_relevancy", "faithfulness", "contextual_recall"],
    "mail":           ["answer_relevancy", "faithfulness", "contextual_recall"],
    "schedule_agent": ["answer_relevancy", "extraction_accuracy"],
    "schedule":       ["answer_relevancy", "extraction_accuracy"],
    "chat_assist":    ["answer_relevancy"],
}

# Phase 1: AI service calls — keep low (3) so the server isn't overwhelmed.
# Phase 2: Metric evals via OpenAI — can be higher since it's a separate service.
MAX_PARALLEL_AI_CALLS    = 3
MAX_PARALLEL_METRIC_EVAL = 15
AI_REQUEST_TIMEOUT_SECONDS = 480.0
STATUS_HEARTBEAT_SECONDS   = 10
RESULTS_DIR    = Path("results")
DEFAULT_RUN_ID = "latest"
RESULTS_LOCK   = Lock()


@contextlib.contextmanager
def _quiet():
    """Suppress all stdout/stderr from DeepEval's rich console output."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        yield


def validate_runtime_config(require_auth: bool = True) -> None:
    missing = []
    if not _cfg("OPENAI_API_KEY"):
        missing.append("OPENAI_API_KEY")
    if not _cfg("AI_SERVICE_URL"):
        missing.append("AI_SERVICE_URL")
    if require_auth and not (_cfg("INTERNAL_API_TOKEN") or _cfg("AUTH_TOKEN")):
        missing.append("INTERNAL_API_TOKEN or AUTH_TOKEN")
    if missing:
        raise RuntimeError("Missing required runtime config: " + ", ".join(missing))


def _headers() -> dict:
    h = {"Content-Type": "application/json"}
    internal_token = _cfg("INTERNAL_API_TOKEN")
    auth_token = _cfg("AUTH_TOKEN")
    if internal_token:
        h["x-internal-api-token"] = internal_token
    if auth_token:
        h["Authorization"] = auth_token if auth_token.startswith("Bearer ") else f"Bearer {auth_token}"
    return h


def _auth_ctx() -> dict:
    return {
        "orgId":       _cfg("ORG_ID"),
        "userId":      _cfg("USER_ID"),
        "orgSlug":     _cfg("ORG_SLUG"),
        "orgName":     _cfg("ORG_NAME"),
        "displayName": _cfg("DISPLAY_NAME"),
        "email":       _cfg("EMAIL"),
    }


def _evaluation_message(user_input: str, feature: str, case: dict) -> str:
    agent_label = AGENT_DISPLAY_NAMES.get(feature, feature)
    parts: list[str] = [
        f"Evaluation target agent: {agent_label} ({feature})",
        "Respond only with the final assistant answer for this request.",
    ]
    retrieval_context = case.get("retrieval_context") or []
    if retrieval_context:
        parts.append("Relevant workspace context:\n" + "\n".join(f"- {ctx}" for ctx in retrieval_context))
    conversation_context = case.get("conversation_context") or []
    if conversation_context:
        conversation = "\n".join(
            f"{msg.get('role', 'user')}: {msg.get('content', '')}"
            for msg in conversation_context
        )
        parts.append("Conversation context:\n" + conversation)
    if feature in ("wiki_agent", "wiki") and (case.get("page_title") or case.get("page_content_markdown")):
        parts.append(f"Wiki page title: {case.get('page_title', 'Eval Page')}")
        parts.append("Current wiki markdown:\n" + case.get("page_content_markdown", ""))
    parts.append("User request:\n" + user_input)
    return "\n\n".join(parts)


def call_ai(user_input: str, feature: str, case: dict, session_id: str) -> tuple[str, int]:
    ai_service_url = _cfg("AI_SERVICE_URL", AI_SERVICE_URL).rstrip("/")
    internal_token = _cfg("INTERNAL_API_TOKEN")
    url = (
        f"{ai_service_url}/api/internal/v1/ai/chat"
        if internal_token
        else f"{ai_service_url}/api/ai/chat"
    )
    payload = {
        "sessionId":   session_id,
        "message":     _evaluation_message(user_input, feature, case),
        "authContext": _auth_ctx(),
        "context":     {"entrypoint": "evaluation", "timeZone": "Asia/Ho_Chi_Minh"},
    }
    print(f"[ai] {feature} session={session_id[-8:]}", flush=True)
    t0 = time.monotonic()
    with httpx.Client(timeout=httpx.Timeout(AI_REQUEST_TIMEOUT_SECONDS, connect=20.0)) as client:
        resp = client.post(url, json=payload, headers=_headers())
        resp.raise_for_status()
    latency_ms = int((time.monotonic() - t0) * 1000)
    print(f"[ai done] {feature} session={session_id[-8:]} latency={latency_ms}ms", flush=True)
    data = resp.json()
    answer = (
        data.get("answer")
        or data.get("content")
        or data.get("message")
        or data.get("data", {}).get("answer")
        or data.get("data", {}).get("content")
        or ""
    )
    return str(answer), latency_ms


_GEVAL_SPECS: dict[str, tuple] = {
    "schedule_agent": (
        "Schedule Action Accuracy",
        "Evaluate whether the output correctly handles creating tasks, events, meetings, or room bookings.",
        [
            "Identify the requested scheduling action.",
            "Check title or subject accuracy, accepting minor paraphrases.",
            "Verify date, time, recurrence, duration, assignee, attendees, priority, room, or dependency when stated.",
            "For underspecified requests, verify the output asks for the missing detail.",
            "Verify the output proposes the action or asks for confirmation where appropriate.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "calendar_agent": (
        "Calendar Grounding Accuracy",
        "Evaluate whether the output correctly answers or proposes updates for existing calendar events and tasks.",
        [
            "Identify the calendar or task item requested.",
            "Verify the output uses the correct event/task details.",
            "For update/delete/complete requests, verify it proposes the change and asks for confirmation.",
            "Check that it does not fabricate unavailable times, attendees, or rooms.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "wiki_agent": (
        "Wiki Agent Accuracy",
        "Evaluate whether the output fulfills the wiki request.",
        [
            "Identify whether the request is wiki editing, search, summarization, translation, or QA.",
            "Check that the output performs that wiki task.",
            "Verify key facts, commands, links, dates, and markdown constraints are preserved.",
            "For translations, verify target language and meaning.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "contacts_agent": (
        "Contacts Lookup Accuracy",
        "Evaluate whether the output answers a workspace contact or directory question.",
        [
            "Identify the requested person, department, role, owner, email, or phone number.",
            "Verify the output includes the correct directory fact.",
            "Check that it does not invent unrelated contacts.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "chat_agent": (
        "Chat Retrieval Accuracy",
        "Evaluate whether the output correctly searches, recaps, or answers questions about chat messages.",
        [
            "Identify the conversation, channel, sender, or message topic requested.",
            "Verify the output reflects the relevant chat snippets or summary.",
            "Check that speakers and message facts are attributed correctly.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "mail_agent": (
        "Mail Retrieval Accuracy",
        "Evaluate whether the output correctly searches, reads, summarizes, or drafts around email threads.",
        [
            "Identify the mail action: search, list, summarize, find sender, or draft reply.",
            "Verify the output includes the correct subject, sender, or summary.",
            "For draft/send requests, verify it drafts or proposes the email without claiming it was sent unless confirmed.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
}
# Short feature aliases point to the same spec
for _short, _full in [("wiki","wiki_agent"),("chat","chat_agent"),("calendar","calendar_agent"),
                       ("contacts","contacts_agent"),("mail","mail_agent"),("schedule","schedule_agent")]:
    _GEVAL_SPECS[_short] = _GEVAL_SPECS[_full]


def build_metrics(metric_names: list[str], feature: str | None = None) -> list:
    metrics = []
    for name in metric_names:
        if name == "answer_relevancy":
            metrics.append(AnswerRelevancyMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "faithfulness":
            metrics.append(FaithfulnessMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "contextual_recall":
            metrics.append(ContextualRecallMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "extraction_accuracy":
            if feature in _GEVAL_SPECS:
                g_name, criteria, steps, params = _GEVAL_SPECS[feature]
            else:
                g_name = "Extraction Accuracy"
                criteria = "Verify whether key information, entities, and actions are correctly represented."
                steps = [
                    "Check whether the actual output identifies all key entities and data points.",
                    "Ensure no incorrect or fabricated information is extracted.",
                    "Verify the extracted details align with the expected output.",
                ]
                params = [LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
            metrics.append(GEval(
                name=g_name, criteria=criteria, evaluation_steps=steps,
                evaluation_params=params, threshold=0.3, model="gpt-4o-mini",
            ))
    return metrics


# ---------------------------------------------------------------------------
# Checkpoint helpers
# ---------------------------------------------------------------------------
def _result_key(dataset: dict, index: int) -> str:
    return f"{dataset.get('agent', dataset['feature'])}::{index}"


def _case_key(result: dict) -> str:
    return f"{result.get('agent', result.get('feature', 'unknown'))}::{result['index']}"


def _result_path(run_id: str = DEFAULT_RUN_ID) -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    return RESULTS_DIR / f"{run_id}.jsonl"


def load_results(run_id: str = DEFAULT_RUN_ID) -> list[dict]:
    with RESULTS_LOCK:
        path = _result_path(run_id)
        if not path.exists():
            return []
        results_by_key: dict[str, dict] = {}
        for line in path.read_text().splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            if isinstance(row, dict) and row.get("type") == "case_result":
                result = row["result"]
                results_by_key[_case_key(result)] = result
            elif isinstance(row, dict) and "agent" in row and "index" in row:
                results_by_key[_case_key(row)] = row
        return sorted(results_by_key.values(), key=lambda r: (r.get("agent", ""), r.get("index", 0)))


def append_result(result: dict, run_id: str = DEFAULT_RUN_ID) -> None:
    with RESULTS_LOCK:
        path = _result_path(run_id)
        row = {
            "type":     "case_result",
            "run_id":   run_id,
            "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "key":      _case_key(result),
            "result":   result,
        }
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_results(results: list[dict], run_id: str = DEFAULT_RUN_ID) -> None:
    with RESULTS_LOCK:
        path = _result_path(run_id)
        tmp = path.with_suffix(path.suffix + ".tmp")
        with tmp.open("w", encoding="utf-8") as f:
            for result in sorted(results, key=lambda r: (r.get("agent", ""), r.get("index", 0))):
                row = {"type": "case_result", "run_id": run_id,
                       "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
                       "key": _case_key(result), "result": result}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        tmp.replace(path)


# ---------------------------------------------------------------------------
# Phase 1 — AI call only
# ---------------------------------------------------------------------------
def _ai_call_only(dataset: dict, case: dict, index: int) -> dict:
    feature  = dataset["feature"]
    agent    = dataset.get("agent", feature)
    ai_error = None
    try:
        actual_output, latency_ms = call_ai(
            user_input=case["input"], feature=feature, case=case,
            session_id=f"eval-{agent}-{uuid4()}",
        )
    except Exception as exc:
        actual_output = f"[AI ERROR: {exc}]"
        latency_ms    = 0
        ai_error      = str(exc)
    return {
        "index":             index,
        "dataset":           dataset["name"],
        "feature":           feature,
        "agent":             agent,
        "input":             case["input"],
        "expected_output":   case.get("expected_output", ""),
        "retrieval_context": case.get("retrieval_context") or None,
        "actual_output":     actual_output,
        "latency_ms":        latency_ms,
        "ai_error":          ai_error,
    }


# ---------------------------------------------------------------------------
# Phase 2 — metric scoring only (DeepEval output fully suppressed)
# ---------------------------------------------------------------------------
def _score_partial(partial: dict, metric_names: list[str]) -> dict:
    feature  = partial["feature"]
    ai_error = partial.get("ai_error")
    test_case = LLMTestCase(
        input=partial["input"],
        actual_output=partial["actual_output"],
        expected_output=partial["expected_output"] or None,
        retrieval_context=partial["retrieval_context"],
    )
    metrics = build_metrics(metric_names, feature=feature)
    try:
        eval_kwargs: dict = {"test_cases": [test_case], "metrics": metrics}
        if DisplayConfig is not None:
            eval_kwargs["display_config"] = DisplayConfig(print_results=False, show_indicator=False)
        if CacheConfig is not None:
            eval_kwargs["cache_config"] = CacheConfig(write_cache=False, use_cache=False)
        with _quiet():
            try:
                eval_result = evaluate(**eval_kwargs)
            except TypeError:
                eval_result = evaluate(test_cases=[test_case], metrics=metrics)
        metric_data = eval_result.test_results[0].metrics_data if eval_result.test_results else []
    except Exception as exc:
        metric_data = []
        ai_error = ai_error or str(exc)

    passed_values = [m.success for m in metric_data if m.success is not None]
    result = {k: v for k, v in partial.items() if k not in ("ai_error", "retrieval_context")}
    result.update({
        "passed":  bool(passed_values) and all(passed_values),
        "error":   ai_error,
        "metrics": [
            {"name": m.name, "score": m.score, "passed": m.success, "reason": m.reason}
            for m in metric_data
        ],
    })
    return result


# ---------------------------------------------------------------------------
# Single-agent evaluation (two-phase)
# ---------------------------------------------------------------------------
def run_evaluation(
    dataset: dict,
    metric_names: list[str] | None = None,
    ai_workers: int = MAX_PARALLEL_AI_CALLS,
    metric_workers: int = MAX_PARALLEL_METRIC_EVAL,
    run_id: str = DEFAULT_RUN_ID,
    resume: bool = True,
) -> list[dict]:
    validate_runtime_config(require_auth=False)
    feature = dataset["feature"]
    agent   = dataset.get("agent", feature)
    cases   = dataset["cases"]
    names   = metric_names or FEATURE_DEFAULT_METRICS.get(feature, ["answer_relevancy"])

    cached = load_results(run_id) if resume else []
    results_by_key = {_case_key(r): r for r in cached}
    pending = [
        (i + 1, case)
        for i, case in enumerate(cases)
        if not resume or _result_key(dataset, i + 1) not in results_by_key
    ]

    ai_w = max(1, min(ai_workers, len(pending))) if pending else 1
    mt_w = max(1, min(metric_workers, len(pending))) if pending else 1

    print(f"\n{'='*60}")
    print(f"Dataset  : {dataset['name']}")
    print(f"Agent    : {AGENT_DISPLAY_NAMES.get(agent, agent)} ({agent})")
    print(f"Metrics  : {', '.join(names)}")
    print(f"Cases    : {len(cases)}  |  Cached: {len(cases)-len(pending)}  |  Pending: {len(pending)}")
    print(f"Workers  : {ai_w} AI, {mt_w} metric")

    if not pending:
        print(f"[{agent}] all cases already cached.\n")
        return sorted(
            [r for k, r in results_by_key.items() if k.startswith(f"{agent}::")],
            key=lambda r: r["index"],
        )

    print(f"\n[{agent}] Phase 1 — {len(pending)} AI call(s) ({ai_w} workers)...")
    partials: list[dict] = []
    with ThreadPoolExecutor(max_workers=ai_w) as executor:
        futures = {executor.submit(_ai_call_only, dataset, case, index): index for index, case in pending}
        bar = tqdm(total=len(futures), desc=f"{agent} AI", unit="call")
        remaining = set(futures)
        started = time.monotonic()
        while remaining:
            done, remaining = wait(remaining, timeout=STATUS_HEARTBEAT_SECONDS, return_when=FIRST_COMPLETED)
            if not done:
                print(f"[{agent}] {len(remaining)} pending, elapsed={int(time.monotonic()-started)}s", flush=True)
                continue
            for f in done:
                try:
                    p = f.result()
                except Exception as exc:
                    idx = futures[f]
                    p = {"index": idx, "dataset": dataset["name"], "feature": feature, "agent": agent,
                         "input": "", "expected_output": "", "retrieval_context": None,
                         "actual_output": "", "latency_ms": 0, "ai_error": f"Worker error: {exc}"}
                partials.append(p)
                bar.update(1)
                err = f" err={p['ai_error']}" if p.get("ai_error") else ""
                print(f"[{agent}] case {p['index']} done {p['latency_ms']}ms{err}", flush=True)
        bar.close()

    print(f"\n[{agent}] Phase 2 — scoring {len(partials)} case(s) ({mt_w} workers)...")
    with ThreadPoolExecutor(max_workers=mt_w) as executor:
        futures = {executor.submit(_score_partial, p, names): p["index"] for p in partials}
        bar = tqdm(total=len(futures), desc=f"{agent} Score", unit="case")
        for f in as_completed(futures):
            idx = futures[f]
            try:
                result = f.result()
            except Exception as exc:
                partial = next(p for p in partials if p["index"] == idx)
                result = {**{k: v for k, v in partial.items() if k not in ("ai_error", "retrieval_context")},
                          "passed": False, "error": f"Scoring error: {exc}", "metrics": []}
            results_by_key[_case_key(result)] = result
            append_result(result, run_id=run_id)
            bar.update(1)
            status = "PASS" if result.get("passed") else "FAIL"
            err = f" err={result['error']}" if result.get("error") else ""
            print(f"[{agent}] case {result['index']} {status}{err}", flush=True)
        bar.close()

    return sorted(
        [r for k, r in results_by_key.items() if k.startswith(f"{agent}::")],
        key=lambda r: r["index"],
    )


# ---------------------------------------------------------------------------
# All-agents evaluation (two-phase, flat pool)
# ---------------------------------------------------------------------------
def run_all_evaluations(
    datasets: dict[str, dict],
    ai_workers: int = MAX_PARALLEL_AI_CALLS,
    metric_workers: int = MAX_PARALLEL_METRIC_EVAL,
    run_id: str = DEFAULT_RUN_ID,
    resume: bool = True,
) -> None:
    validate_runtime_config(require_auth=False)

    cached = load_results(run_id) if resume else []
    results_by_key: dict[str, dict] = {_case_key(r): r for r in cached}

    all_pending: list[tuple[dict, int, dict, list[str]]] = []
    for _name, dataset in datasets.items():
        feature = dataset["feature"]
        names   = FEATURE_DEFAULT_METRICS.get(feature, ["answer_relevancy"])
        for i, case in enumerate(dataset["cases"]):
            if not resume or _result_key(dataset, i + 1) not in results_by_key:
                all_pending.append((dataset, i + 1, case, names))

    total_cases = sum(len(d["cases"]) for d in datasets.values())
    print(f"{'='*60}")
    print(f"Agents      : {len(datasets)}  ({', '.join(AGENT_DISPLAY_NAMES.get(k, k) for k in datasets)})")
    print(f"Total cases : {total_cases}  |  Cached: {total_cases - len(all_pending)}  |  Pending: {len(all_pending)}")
    print(f"AI workers  : {ai_workers}  |  Metric workers: {metric_workers}")
    print(f"Timeout     : {AI_REQUEST_TIMEOUT_SECONDS}s per AI call")
    print(f"Result file : {_result_path(run_id)}")

    if not all_pending:
        print("All cases already cached — load results with load_results().")
        return

    print(f"\nPhase 1 — {len(all_pending)} AI call(s) ({ai_workers} workers)...")
    partials: list[tuple[dict, list[str], dict]] = []
    with ThreadPoolExecutor(max_workers=min(ai_workers, len(all_pending))) as executor:
        future_to_meta = {
            executor.submit(_ai_call_only, dataset, case, index): (dataset, names)
            for dataset, index, case, names in all_pending
        }
        bar = tqdm(total=len(future_to_meta), desc="Phase 1 AI", unit="call")
        remaining = set(future_to_meta)
        started = time.monotonic()
        while remaining:
            done, remaining = wait(remaining, timeout=STATUS_HEARTBEAT_SECONDS, return_when=FIRST_COMPLETED)
            if not done:
                active = min(ai_workers, len(remaining))
                queued = max(0, len(remaining) - active)
                print(f"[Phase 1] {active} active / {queued} queued, elapsed={int(time.monotonic()-started)}s", flush=True)
                continue
            for f in done:
                dataset, names = future_to_meta[f]
                try:
                    p = f.result()
                except Exception as exc:
                    p = {"index": -1, "dataset": dataset["name"],
                         "feature": dataset["feature"], "agent": dataset.get("agent", dataset["feature"]),
                         "input": "", "expected_output": "", "retrieval_context": None,
                         "actual_output": "", "latency_ms": 0, "ai_error": f"Worker error: {exc}"}
                partials.append((dataset, names, p))
                bar.update(1)
                err = f" err={p['ai_error']}" if p.get("ai_error") else ""
                print(f"[Phase 1] {p['agent']} case {p['index']} {p['latency_ms']}ms{err}", flush=True)
        bar.close()

    print(f"\nPhase 2 — scoring {len(partials)} case(s) ({metric_workers} workers)...")
    with ThreadPoolExecutor(max_workers=min(metric_workers, len(partials))) as executor:
        future_to_partial = {
            executor.submit(_score_partial, p, names): p
            for _dataset, names, p in partials
        }
        bar = tqdm(total=len(future_to_partial), desc="Phase 2 Score", unit="case")
        for f in as_completed(future_to_partial):
            partial = future_to_partial[f]
            try:
                result = f.result()
            except Exception as exc:
                result = {**{k: v for k, v in partial.items() if k not in ("ai_error", "retrieval_context")},
                          "passed": False, "error": f"Scoring error: {exc}", "metrics": []}
            results_by_key[_case_key(result)] = result
            append_result(result, run_id=run_id)
            bar.update(1)
            status = "PASS" if result.get("passed") else "FAIL"
            err = f" err={result['error']}" if result.get("error") else ""
            print(f"[Phase 2] {result['agent']} case {result['index']} {status}{err}", flush=True)
        bar.close()

    saved = len(load_results(run_id))
    print(f"\nDone — {saved} result(s) saved to {_result_path(run_id)}")


# ---------------------------------------------------------------------------
# Display helpers
# ---------------------------------------------------------------------------
def display_results(results: list[dict], title: str | None = None) -> None:
    total    = len(results)
    passed   = sum(1 for r in results if r["passed"])
    failed   = total - passed
    pass_pct = passed / total * 100 if total else 0

    metric_stats: dict[str, dict] = {}
    for r in results:
        for m in r["metrics"]:
            s = metric_stats.setdefault(m["name"], {"total": 0, "sum": 0.0, "passed": 0})
            if m["score"] is not None:
                s["total"] += 1
                s["sum"]   += m["score"]
                if m["passed"]:
                    s["passed"] += 1

    title_html = f'<h3 style="font-family:monospace;margin:0 0 10px;color:#0f172a">{escape(title)}</h3>' if title else ""
    cards = "".join([
        f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:12px 18px;min-width:90px"><div style="font-size:24px;font-weight:700;color:#16a34a">{passed}</div><div style="font-size:10px;color:#64748b;text-transform:uppercase">Passed</div></div>',
        f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:12px 18px;min-width:90px"><div style="font-size:24px;font-weight:700;color:#dc2626">{failed}</div><div style="font-size:10px;color:#64748b;text-transform:uppercase">Failed</div></div>',
        f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:12px 18px;min-width:90px"><div style="font-size:24px;font-weight:700;color:#0f172a">{total}</div><div style="font-size:10px;color:#64748b;text-transform:uppercase">Total</div></div>',
        f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:12px 18px;min-width:90px"><div style="font-size:24px;font-weight:700;color:#0f172a">{pass_pct:.0f}%</div><div style="font-size:10px;color:#64748b;text-transform:uppercase">Pass rate</div></div>',
    ])

    metric_rows = ""
    for name, s in metric_stats.items():
        avg   = s["sum"] / s["total"] if s["total"] else 0
        pct   = int(avg * 100)
        p_rt  = int(s["passed"] / s["total"] * 100) if s["total"] else 0
        color = "#22c55e" if pct >= 70 else "#f59e0b" if pct >= 50 else "#ef4444"
        metric_rows += (
            '<div style="display:flex;align-items:center;gap:12px;margin:6px 0">'
            f'<span style="width:180px;font-size:11px;color:#475569">{escape(name.replace("_", " "))}</span>'
            '<div style="width:180px;height:6px;background:#e2e8f0;border-radius:999px;overflow:hidden">'
            f'<div style="height:100%;background:{color};width:{pct}%"></div></div>'
            f'<span style="font-size:11px;font-weight:700;color:{color};width:38px">{pct}%</span>'
            f'<span style="font-size:10px;color:#94a3b8">{p_rt}% pass</span></div>'
        )

    rows = ""
    for r in results:
        sc = "#16a34a" if r["passed"] else "#dc2626"
        sl = "pass" if r["passed"] else "fail"
        badges = ""
        for m in r["metrics"]:
            if m["score"] is None:
                badges += f'<span style="font-size:9px;color:#64748b;background:#f1f5f9;border-radius:4px;padding:2px 6px;margin:1px">{escape(m["name"])}: -</span>'
            else:
                pct = int(m["score"] * 100)
                mc  = "#22c55e" if pct >= 70 else "#f59e0b" if pct >= 50 else "#ef4444"
                badges += f'<span style="font-size:9px;color:{mc};background:{mc}18;border-radius:4px;padding:2px 6px;margin:1px">{escape(m["name"].replace("_", " "))}: {pct}%</span>'
        err = f'<div style="font-size:9px;color:#dc2626;margin-top:3px">{escape(str(r["error"]))}</div>' if r.get("error") else ""
        rows += (
            '<tr style="border-bottom:1px solid #f1f5f9">'
            f'<td style="padding:8px;color:#94a3b8;font-size:11px">#{r["index"]}</td>'
            f'<td style="padding:8px;color:#334155;font-size:11px;max-width:260px">{escape(str(r["input"]))}</td>'
            f'<td style="padding:8px;color:#334155;font-size:11px;max-width:360px">{escape(str(r["actual_output"]))[:240]}</td>'
            f'<td style="padding:8px"><span style="font-size:10px;font-weight:700;color:{sc}">{sl}</span><div>{badges}</div>{err}</td>'
            f'<td style="padding:8px;color:#94a3b8;font-size:10px;white-space:nowrap">{r["latency_ms"]}ms</td>'
            '</tr>'
        )

    html = (
        '<div style="font-family:monospace;margin:10px 0 18px">'
        + title_html
        + f'<div style="display:flex;gap:12px;margin-bottom:12px">{cards}</div>'
        + (f'<div style="border:1px solid #e2e8f0;border-radius:8px;padding:12px;margin-bottom:12px">{metric_rows}</div>' if metric_rows else "")
        + '<table style="width:100%;border-collapse:collapse"><thead><tr style="background:#f8fafc">'
        + '<th style="padding:8px;text-align:left;font-size:9px;color:#64748b">#</th>'
        + '<th style="padding:8px;text-align:left;font-size:9px;color:#64748b">Input</th>'
        + '<th style="padding:8px;text-align:left;font-size:9px;color:#64748b">Actual Output</th>'
        + '<th style="padding:8px;text-align:left;font-size:9px;color:#64748b">Result</th>'
        + '<th style="padding:8px;text-align:left;font-size:9px;color:#64748b">Latency</th>'
        + f'</tr></thead><tbody>{rows}</tbody></table></div>'
    )
    display(HTML(html))


def display_results_by_agent(results: list[dict]) -> None:
    grouped: dict[str, list[dict]] = {}
    for result in results:
        grouped.setdefault(result.get("agent", result.get("feature", "unknown")), []).append(result)
    for agent in sorted(grouped):
        label = AGENT_DISPLAY_NAMES.get(agent, agent)
        display_results(grouped[agent], title=f"{label} — {len(grouped[agent])} case(s)")


def check_ai_api_once(dataset_name: str | None = None) -> None:
    validate_runtime_config(require_auth=False)
    dataset    = datasets[dataset_name] if dataset_name else next(iter(datasets.values()))
    case       = dataset["cases"][0]
    feature    = dataset["feature"]
    session_id = f"debug-{feature}-{uuid4()}"
    ai_service_url = _cfg("AI_SERVICE_URL", AI_SERVICE_URL).rstrip("/")
    internal_token = _cfg("INTERNAL_API_TOKEN")
    url = (
        f"{ai_service_url}/api/internal/v1/ai/chat"
        if internal_token
        else f"{ai_service_url}/api/ai/chat"
    )
    payload = {
        "sessionId":   session_id,
        "message":     _evaluation_message(case["input"], feature, case),
        "authContext": _auth_ctx(),
        "context":     {"entrypoint": "evaluation-debug", "timeZone": "Asia/Ho_Chi_Minh"},
    }
    print(f"[preflight] URL: {url}")
    print(f"[preflight] feature: {feature}  session: {session_id}")
    print(f"[preflight] INTERNAL_API_TOKEN: {bool(_cfg('INTERNAL_API_TOKEN'))}  AUTH_TOKEN: {bool(_cfg('AUTH_TOKEN'))}")
    t0 = time.monotonic()
    try:
        with httpx.Client(timeout=httpx.Timeout(45.0, connect=10.0), follow_redirects=False) as client:
            response = client.post(url, json=payload, headers=_headers())
        elapsed_ms = int((time.monotonic() - t0) * 1000)
        print(f"[preflight] status={response.status_code} elapsed={elapsed_ms}ms")
        print(f"[preflight] response: {response.text[:1000]}")
        response.raise_for_status()
    except Exception as exc:
        elapsed_ms = int((time.monotonic() - t0) * 1000)
        print(f"[preflight] FAILED after {elapsed_ms}ms: {type(exc).__name__}: {exc}")
        raise


print("Setup complete. Agents:", ", ".join(AGENT_DISPLAY_NAMES.values()))
print(f"AI workers: {MAX_PARALLEL_AI_CALLS}  |  Metric workers: {MAX_PARALLEL_METRIC_EVAL}  |  Timeout: {AI_REQUEST_TIMEOUT_SECONDS}s")

Setup complete. Agents: Wiki Agent, Chat Agent, Calendar Agent, Contacts Agent, Mail Agent, Schedule Agent, Chat Assist
AI workers: 3  |  Metric workers: 15  |  Timeout: 480.0s


## 3 · Load Datasets

Scans the `datasets/` folder and prints all available test suites.

In [4]:
DATASETS_DIR = Path("datasets")

datasets: dict[str, dict] = {}
for _path in sorted(DATASETS_DIR.glob("*.json")):
    _d = json.loads(_path.read_text())
    datasets[_d["name"]] = _d

agent_names = sorted({_d.get("agent", _d["feature"]) for _d in datasets.values()})
print(f"Found {len(datasets)} dataset(s) covering {len(agent_names)} chat-routed agent(s):")
print("  " + ", ".join(AGENT_DISPLAY_NAMES.get(a, a) for a in agent_names) + "\n")

for _name, _d in datasets.items():
    _agent = _d.get("agent", _d["feature"])
    _metrics = FEATURE_DEFAULT_METRICS.get(_d["feature"], ["answer_relevancy"])
    print(f"  {_name}")
    print(f"    agent    : {AGENT_DISPLAY_NAMES.get(_agent, _agent)} ({_agent})")
    print(f"    feature  : {_d['feature']}")
    print(f"    cases    : {len(_d['cases'])}")
    print(f"    metrics  : {', '.join(_metrics)}\n")


Found 7 dataset(s) covering 7 chat-routed agent(s):
  Calendar Agent, Chat Agent, Chat Assist, Contacts Agent, Mail Agent, Schedule Agent, Wiki Agent

  Calendar Agent Evaluation
    agent    : Calendar Agent (calendar_agent)
    feature  : calendar
    cases    : 10
    metrics  : answer_relevancy, faithfulness, contextual_recall, extraction_accuracy

  Chat Agent Evaluation
    agent    : Chat Agent (chat_agent)
    feature  : chat
    cases    : 10
    metrics  : answer_relevancy, faithfulness, contextual_recall

  Chat Assist Evaluation
    agent    : Chat Assist (chat_assist)
    feature  : chat_assist
    cases    : 12
    metrics  : answer_relevancy

  Contacts Agent Evaluation
    agent    : Contacts Agent (contacts_agent)
    feature  : contacts
    cases    : 10
    metrics  : answer_relevancy, faithfulness, contextual_recall

  Mail Agent Evaluation
    agent    : Mail Agent (mail_agent)
    feature  : mail
    cases    : 10
    metrics  : answer_relevancy, faithfulness, con

## 4 · Run All Agents

Runs every agent dataset and renders results grouped by agent.

In [7]:
# 3 concurrent AI calls to avoid overwhelming the service.
# 480s timeout gives slow agents (calendar, wiki) enough room.
# resume=True skips already-completed cases — rerun safely on interruption.
run_all_evaluations(
    datasets,
    ai_workers=3,
    metric_workers=MAX_PARALLEL_METRIC_EVAL,
    run_id=DEFAULT_RUN_ID,
    resume=True,
)

Agents      : 7  (Calendar Agent Evaluation, Chat Agent Evaluation, Chat Assist Evaluation, Contacts Agent Evaluation, Mail Agent Evaluation, Schedule Agent Evaluation, Wiki Agent Evaluation)
Total cases : 80  |  Cached: 80  |  Pending: 0
AI workers  : 3  |  Metric workers: 15
Timeout     : 480.0s per AI call
Result file : results/latest.jsonl
All cases already cached — load results with load_results().


## Load Saved Results

Loads the latest checkpoint without calling the API again.

In [8]:
saved_results = load_results(DEFAULT_RUN_ID)
display_results_by_agent(saved_results)


#,Input,Actual Output,Result,Latency
#1,What events do I have today?,"You have one event today: **Daily Standup** from **3:00 AM to 3:30 AM UTC**. It’s a meeting with **Huy Phan, Bob Chen, David Miller, Mary Watson, and John Park**.",passAnswer Relevancy: 78%,6551ms
#2,What is happening on next Monday?,"Next Monday, there are three meetings on the calendar: Weekly Team Standup from 2:00 AM to 3:00 AM UTC with 9 attendees, Sprint 13 Planning from 4:00 AM to 5:30 AM UTC with Huy Phan, Alex Vance, Bob Chen, David Miller, Mary Watson, and John",passAnswer Relevancy: 80%,7484ms
#3,Who is attending the Design Review this Friday?,"The Design Review — Sprint 12 this Friday is scheduled with Huy Phan, Emily Rose, John Park, and Alex Vance.",passAnswer Relevancy: 88%,6642ms
#4,What room is Sprint 13 Planning in?,Sprint 13 Planning is in Room B.,passAnswer Relevancy: 96%,5572ms
#5,When is the Architecture Review for the AI Gateway Migration?,"The Architecture Review: AI Gateway Migration is scheduled for June 24, 2026, from 8:00 AM to 9:00 AM UTC. It includes Alex Vance, Bob Chen, David Miller, and Huy Phan.",passAnswer Relevancy: 78%,5785ms
#6,What are my high-priority tasks that are in progress?,"Your high-priority tasks that are currently in progress are: - Integrate G-Eval metric into evaluation pipeline — Assignee: Huy Phan, Due: Friday - Add gateway proxy config for AI service — Assignee: Bob Chen, Due: Friday",passAnswer Relevancy: 96%,6356ms
#7,What tasks are assigned to Bob Chen?,"Bob Chen is assigned to two workspace tasks: - Add gateway proxy config for AI service — High priority, In progress, due Friday - Add rate limiting to AI service endpoint — High priority, Todo, due next Tuesday From the calendar data, Bob",passAnswer Relevancy: 84%,9177ms
#8,Mark the task 'Publish changelog for v2.3.0' as done.,"I couldn’t find a calendar item matching “Publish changelog for v2.3.0,” so I’m not able to mark it done from the calendar data. The workspace context does show a task with that title, assigned to Huy Phan, currently TODO and due tomorrow,",passAnswer Relevancy: 50%,7631ms
#9,When is the Company All-Hands and what room is it in?,"The Company All-Hands & Product Demo is scheduled for June 25, 2026, from 7:00 AM to 8:30 AM UTC. The raw calendar data does not list a room for this event.",passAnswer Relevancy: 52%,6489ms
#10,List all tasks that are marked as DONE.,The task marked as DONE in the calendar data is: - Share evaluation report with Alex — scheduled for 2026-06-19 10:00 UTC,failAnswer Relevancy: 25%,6002ms


#,Input,Actual Output,Result,Latency
#1,Who announced that Sprint 12's goal is the AI evaluation pipeline?,"Alex Vance announced that Sprint 12’s goal is to get the AI evaluation pipeline to a stable, demo-able state. He also said all seven agents should be scoring above 75% by the end of the sprint.",passAnswer Relevancy: 88%,7490ms
#2,What did Bob say about v2.2.0 in #general?,"Bob said that v2.2.0 was deployed to staging and that the auth flow was stable with smoke tests passing. He later said it was live on production with zero incidents, and that the rollback window would remain open until 6 PM if anything came",passAnswer Relevancy: 88%,18091ms
#3,What bugs did Mary report after QA on v2.2.0?,"Mary reported two bugs after QA on v2.2.0: the wiki editor loses cursor position after autosave in Firefox, and the calendar event modal does not close when pressing Escape. She also said tasks were created for John and Bob respectively.",passAnswer Relevancy: 86%,8267ms
#4,What evaluation scores did Huy post in #general?,"Huy posted two updates in #general: - First pass evaluation: wiki 78%, chat 81%, contacts 91%, calendar 82%, schedule 87%. - Later update: contacts 91%, mail 76%, and he said all agents are now green.",passAnswer Relevancy: 94%,7982ms
#5,What did Sarah announce about Q2 results in #general?,"Sarah announced that Q2 had 2,300 new signups, which was 14% above target. She also said retention was 68%, up 5 points from Q1, and described it as the team’s best quarter so far.",passAnswer Relevancy: 92%,5854ms
#6,What did David post about the race condition in #engineering?,"David said he found the root cause of the token refresh race condition: two concurrent `POST /api/auth/refresh` requests were both reading and validating the current token, then both trying to write a new one, so only the second write stuck",passAnswer Relevancy: 87%,25199ms
#7,What is Alex's proposal for the AI service in Sprint 13?,"Alex’s proposal for Sprint 13 is to move the AI service behind the gateway so it has no direct external access to port 8000. Under this setup, all AI requests would go through `/api/ai/*` on the gateway, and the eval notebook would also rou",passAnswer Relevancy: 92%,8402ms
#8,What did Sarah and Alice discuss about the Q3 campaign CTA in #marketing?,"Sarah said the Q3 campaign brief is ready for #marketing and asked Alice and Emily to review it by end of day Friday. Alice responded that the CTA should be changed from “Get started” to “Start free,” noting that in past A/B tests the “free",passAnswer Relevancy: 72%,6585ms
#9,Summarize the DM conversation between me and Sarah.,"Sarah asked if you were available for a quick call to review the Q3 board presentation before she sends it to Linda Park at Venture X, and you confirmed 2 PM works. She then asked for the Q3 AI infrastructure cost, and you shared that it’s",passAnswer Relevancy: 90%,9197ms
#10,What did Alice say about the landing page in our DMs?,"In your DMs, Alice said she would update the landing page copy and push a new build to staging. She also said she’d add the testimonial to the testimonials section with Michael’s name and title, and that staging would be ready in about 20 m",passAnswer Relevancy: 65%,6664ms


#,Input,Actual Output,Result,Latency
#1,Translate this to Vietnamese: 'The sprint goal is to complete the AI evaluation pipeline. All agents must be above 75% by end of sprint.',Mục tiêu của sprint là hoàn thành quy trình đánh giá AI. Tất cả các tác nhân phải đạt trên 75% vào cuối sprint.,passAnswer Relevancy: 90%,3017ms
#2,Translate this to English: 'Chúng tôi đã triển khai phiên bản v2.2.0 lên môi trường staging và tất cả các bài kiểm thử khói đều đã qua.',"We have deployed version v2.2.0 to the staging environment, and all smoke tests have passed.",passAnswer Relevancy: 96%,2870ms
#3,"Translate this to French: 'Your whole team, one place. Serenity helps teams communicate and ship faster.'","Votre équipe au complet, au même endroit. Serenity aide les équipes à communiquer et à livrer plus rapidement.",passAnswer Relevancy: 92%,2822ms
#4,Fix the grammar in this message: 'We has been working on the evaluation pipeline since last sprint and the result is showing great improvement across all of the agent we tested.',"We have been working on the evaluation pipeline since last sprint, and the results are showing great improvement across all of the agents we tested.",passAnswer Relevancy: 97%,2161ms
#5,"Make this message sound more professional: 'hey just wanted to check if the pr is done yet, been waiting a bit'","Hi, I just wanted to check whether the PR is finished yet. I’ve been waiting for a bit and wanted to follow up.",passAnswer Relevancy: 82%,3780ms
#6,Make this message warmer and more encouraging: 'QA is complete. Two bugs were found. They will be fixed before release.',"QA is complete, and the team identified two bugs that will be fixed before release. Great progress overall!",passAnswer Relevancy: 90%,2139ms
#7,Make this more direct and concise: 'I was wondering if it might be possible for us to perhaps consider scheduling a meeting at some point next week to go over the Q3 OKRs together if everyone is available and it works for the team.',Could we schedule a meeting next week to review the Q3 OKRs if everyone is available?,passAnswer Relevancy: 92%,3156ms
#8,Rephrase this update so it sounds more enthusiastic: 'Evaluation results are in. Contacts agent scored 91%. All agents are above the 75% threshold.',"Great news—evaluation results are in! The Contacts agent scored 91%, and every agent is performing above the 75% threshold.",passAnswer Relevancy: 91%,2591ms
#9,Draft a polite message to decline a meeting invitation for next Monday's 9 AM Weekly Team Standup because I have a conflict.,"Hi team, thank you for the invitation. Unfortunately, I have a conflict and won’t be able to make next Monday’s 9:00 AM Weekly Team Standup. I apologize for the inconvenience and hope to catch up on anything I miss.",passAnswer Relevancy: 94%,3691ms
#10,Draft a brief status update message to send to #general saying that the evaluation notebook is now stable and all 7 agents are green.,"Quick update: the evaluation notebook is now stable, and all 7 agents are green.",passAnswer Relevancy: 88%,2784ms


#,Input,Actual Output,Result,Latency
#1,Who is on the Engineering team?,"The Engineering team includes Alex Vance (Engineering Lead), Bob Chen (Senior Software Engineer), David Miller (Backend Engineer), Mary Watson (QA Engineer), John Park (Frontend Engineer), and Huy Phan (creator/owner).",passAnswer Relevancy: 86%,6696ms
#2,What is Alice Nguyen's email address?,Alice Nguyen’s email address is alice.nguyen-vku@example.com.,passAnswer Relevancy: 94%,5429ms
#3,What is David Miller's phone number?,David Miller’s phone number is +1-555-0148.,passAnswer Relevancy: 98%,7374ms
#4,What is Sarah Connor's title and email?,"Sarah Connor’s title is Head of Marketing, and her email is sarah.connor@workspace.com.",passAnswer Relevancy: 98%,5399ms
#5,Who is the Engineering Lead and what is their phone number?,"The Engineering Lead is Alex Vance, and their phone number is +1-555-0175.",passAnswer Relevancy: 96%,5285ms
#6,What are the email and phone number for Emily Rose?,Emily Rose’s email is emily.rose@workspace.com and her phone number is +1-555-0197.,passAnswer Relevancy: 98%,5623ms
#7,Who are the members of the Marketing department?,"The Marketing department has two members: Alice Nguyen, Marketing Manager, and Sarah Connor, Head of Marketing.",passAnswer Relevancy: 98%,5655ms
#8,How can I contact Michael Torres from NovaTech Solutions?,You can reach Michael Torres at michael.torres@novatech.io or by phone at +1-555-0230. He is the Head of Engineering at NovaTech Solutions.,passAnswer Relevancy: 96%,5273ms
#9,What is Linda Park's role and how do I reach her?,Linda Park is a Partner at Venture X Capital. You can reach her by email at linda.park@venture-x.com or by phone at +1-555-0241.,passAnswer Relevancy: 98%,4514ms
#10,Find the contact details for James Okafor.,James Okafor’s contact details are: - Email: j.okafor@designhub.co - Phone: +1-555-0259 - Company: DesignHub - Title: Freelance Consultant,passAnswer Relevancy: 98%,5704ms


#,Input,Actual Output,Result,Latency
#1,Find emails about the Q3 board presentation.,I couldn’t find any mail threads matching “Q3 board presentation.”,failAnswer Relevancy: 20%,5968ms
#2,Show me recent emails from Alice Nguyen.,I couldn’t find any mail threads in the mail agent data. The workspace context shows two recent emails from Alice Nguyen to huy.phan@workspace.com: - **Subject:** Landing Page Copy — Staging Review **Date:** Tuesday She says the ne,passAnswer Relevancy: 78%,8785ms
#3,Find emails from Michael Torres.,I couldn’t find any mail threads matching “Michael Torres” in the retrieved mail data.,failAnswer Relevancy: 20%,18769ms
#4,Summarize the email thread about the Redis graceful fallback PR.,PR #52 for the Redis session layer was reviewed and updated after Bob flagged that Redis failures were returning a 500 instead of falling back to the DB session store. Huy changed it so Redis failure now falls back to the Prisma session sto,passAnswer Relevancy: 92%,6034ms
#5,"Draft a reply to Sarah's email about the Q3 board presentation, confirming the AI infrastructure cost is $2,400/month covering GPU instances and OpenAI API costs, and that it will drop 30% when we switch to our own model.","Hi Sarah, Yes — the AI infrastructure cost for Q3 is $2,400/month, which covers GPU instances and OpenAI API costs. Once we switch to our own model, I expect that cost to drop by about 30%. Best, Huy",passAnswer Relevancy: 94%,5646ms
#6,Draft a new email to Linda Park at Venture X Capital introducing Serenity and requesting a 30-minute introductory call.,"I can draft the email to Linda Park at Venture X Capital requesting a 30-minute introductory call, but the mail action returned a 400 Bad Request error, so I couldn’t send anything yet. Please confirm if you’d like me to try again once the",failAnswer Relevancy: 35%,5930ms
#7,Search for emails related to the September 15 product launch.,I couldn’t find any mail threads matching “September 15 product launch.”,failAnswer Relevancy: 20%,4894ms
#8,Find emails about the NovaTech testimonial.,"I couldn’t find any mail threads matching “NovaTech testimonial” or “testimonial” in the retrieved mail results. However, the workspace context includes an email from Michael Torres at NovaTech Solutions titled “Testimonial Permission — Nov",passAnswer Relevancy: 78%,9325ms
#9,Draft an email to James Okafor at DesignHub asking about his availability for a freelance design consultation next week.,"I drafted the email to James Okafor at DesignHub asking about his availability for a freelance design consultation next week, but I couldn’t send it because the mail service returned a 400 Bad Request error. If you want, I can revise the dr",failAnswer Relevancy: 35%,6239ms
#10,Show me all emails I received this week.,"I couldn’t find any email threads in mail agent results. Based on the workspace context, you received these emails this week: - Monday — from sarah.connor@workspace.com: “Q3 Board Presentation — AI Infrastructure Cost” - Tuesday — from ali",passAnswer Relevancy: 76%,5678ms


#,Input,Actual Output,Result,Latency
#1,"Create a high priority task for me to write the Q3 performance report, due next Friday.",I’m ready to create a high-priority task titled “Write Q3 performance report” due on 2026-06-26. Please confirm if you’d like me to proceed.,passAnswer Relevancy: 55%,4813ms
#2,"Create a medium priority task for Emily Rose to finalize the onboarding flow mockups, due next Wednesday.","I’m ready to create a medium-priority task for Emily Rose titled “Finalize the onboarding flow mockups,” due next Wednesday. The proposed due date is 2026-06-24, and the task would be assigned to Emily Rose. Please confirm if you’d like me",passAnswer Relevancy: 55%,5651ms
#3,Schedule a 30-minute meeting with Bob Chen and David Miller tomorrow at 2 PM to discuss the Redis session layer.,"I’m ready to schedule a 30-minute meeting titled “Discuss Redis session layer” for tomorrow, June 19, 2026, from 2:00 PM to 2:30 PM with Bob Chen and David Miller. Please confirm if you’d like me to create it.",passAnswer Relevancy: 62%,7113ms
#4,Book Room A for a design workshop next Thursday from 10 AM to 12 PM.,I can book Room A for a Design workshop on 2026-06-25 from 10:00 AM to 12:00 PM. Please confirm if you’d like me to proceed with this room booking.,passAnswer Relevancy: 56%,7244ms
#5,Set up a weekly recurring 1-on-1 meeting with Mary Watson every Tuesday at 11 AM for 30 minutes.,"I can set up a 30-minute 1-on-1 with Mary Watson for Tuesday at 11:00 AM, but the available action only shows a one-time meeting on June 23, 2026 from 11:00 AM to 11:30 AM. I don’t have a recurring-weekly rule in the proposed action, so ple",failAnswer Relevancy: 42%,6008ms
#6,"Create two tasks: one high priority task for me to prepare the Q3 board slide deck due next Monday, and one medium priority task for Alice to draft the September blog post due next Wednesday.","I can create the high-priority task for you: “Prepare Q3 board slide deck,” due Monday, June 22, 2026. I don’t have enough information to create the second task yet because there are multiple Alice Nguyen contact entries with the same ema",passAnswer Relevancy: 50%,5491ms
#7,Remind me to follow up on PR #52 feedback next Monday at 9 AM.,"I can set a reminder/task for “Follow up on PR #52 feedback,” but the available scheduling data only gives a due date of 2026-06-22 and doesn’t include “next Monday at 9 AM.” Please confirm if you want me to create it with the due date 2026",failAnswer Relevancy: 45%,5738ms
#8,Schedule a meeting.,Sure — what meeting title and time should I propose?,passAnswer Relevancy: 60%,4429ms
#9,"Create a low priority task to update the README for the ai-service, due next Friday.",I can create a low-priority task titled “Update README for ai-service” with a due date of 2026-06-26. Please confirm if you’d like me to proceed.,passAnswer Relevancy: 55%,6116ms
#10,Book the Main Conference Room for the all-hands rehearsal next Wednesday from 1 PM to 2 PM.,I’m ready to book the Main Conference Room for the all-hands rehearsal on 2026-06-24 from 1:00 PM to 2:00 PM. Please confirm if you’d like me to proceed.,passAnswer Relevancy: 56%,6343ms


#,Input,Actual Output,Result,Latency
#1,What port does the auth-service run on and who owns it?,The auth-service runs on port 3001 and is owned by David Miller.,passAnswer Relevancy: 98%,5603ms
#2,Who owns the core-service and what port does it use?,The core-service is owned by Alex Vance and uses port 2993.,passAnswer Relevancy: 98%,7731ms
#3,What is the on-call rotation for Sprint 13?,"For Sprint 13, the primary on-call is David Miller and the backup is John Park.",passAnswer Relevancy: 98%,3895ms
#4,What fields must be included in every PR description?,"Every PR description must include these sections: **Problem**, **Solution**, **Verification**, **Risk**, and **Screenshots** if the change affects any UI.",passAnswer Relevancy: 92%,10589ms
#5,Who can approve production deployments?,Production deployments require approval from either Alex Vance or David Miller before merging.,passAnswer Relevancy: 93%,6929ms
#6,What is the default password for seed users in development?,I cannot return that response safely.,failAnswer Relevancy: 35%,7867ms
#7,What is the brand tagline and what is the primary brand color token?,"The brand tagline is “Your whole team, one place.” The primary brand color token is `--brand` (a deep navy, `#070738`).",passAnswer Relevancy: 97%,7378ms
#8,Summarize the Engineering Delivery Guide wiki page.,"The Engineering Delivery Guide explains how engineering work should move from idea to shipped change, with an emphasis on small, reviewable PRs, using existing architecture, keeping behavior/tests/docs together when relevant, and ensuring f",passAnswer Relevancy: 88%,12856ms
#9,"Add a checklist section to this page for first-week tasks: set up local dev environment, join #general and #engineering channels, complete security training, have 1-on-1 with manager.","I couldn’t find the wiki page you referenced, so I can’t add the checklist section yet. If you send the correct page or link, I can update it with a first-week checklist for: setting up the local dev environment, joining #general and #engin",failAnswer Relevancy: 35%,5014ms
#10,Add a service map table listing all five services with their port and owner.,"I can’t make that edit right now because no wiki page is currently open. If you open the target wiki page, I can add a service map table listing the five services with their ports and owners.",failAnswer Relevancy: 25%,8476ms
